In [ ]:
from ultralytics import YOLO
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType
import os
import datetime

In [ ]:
# Configuration

# Data YAML path
DATA_YAML = "data/data.yaml"

# Base model for fine-tuning
BASE_MODEL = "yolov8n.pt"

# Training settings
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = "0"  # "0" for first GPU, "cpu" for CPU
PROJECT = "runs/train"
NAME = "detect_boxs"

# Resume training if a run with the same name exists
RESUME_IF_EXISTS = True

# Export filenames
ONNX_NAME = "best_box_detector.onnx"
ONNX_INT8_NAME = "best_box_detector_int8.onnx"

print("DATA_YAML:", DATA_YAML)
print("BASE_MODEL:", BASE_MODEL)


In [ ]:
# Cell 3: Helpers to find last/best run

def get_latest_run(project: str) -> Path | None:
    # Return the most recently modified run directory inside project, or None
    p = Path(project)
    if not p.exists():
        return None
    runs = [d for d in p.iterdir() if d.is_dir()]
    if not runs:
        return None
    return sorted(runs, key=lambda x: x.stat().st_mtime)[-1]

def find_best_pt(run_dir: Path) -> Path | None:
    # Return path to best.pt inside run_dir/weights if it exists, else None
    cand = run_dir / "weights" / "best.pt"
    return cand if cand.exists() else None


In [ ]:
# Training with checkpointing and optional resume

exp_dir = Path(PROJECT) / NAME
latest_run = get_latest_run(PROJECT)

resume_flag = False
if RESUME_IF_EXISTS and latest_run is not None and latest_run.name == NAME:
    print(f"[INFO] Found existing run: {latest_run}, will resume.")
    resume_flag = True
else:
    print("[INFO] Starting new run.")

model = YOLO(BASE_MODEL)

train_args = dict(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=PROJECT,
    name=NAME,
    save=True,
    save_period=1,
    exist_ok=True,
)

if resume_flag:
    results = model.train(resume=True, **train_args)
else:
    results = model.train(**train_args)

print("[INFO] Training finished.")


In [ ]:
# Export best.pt to ONNX

run_dir = get_latest_run(PROJECT)
if run_dir is None:
    raise RuntimeError("No run directory found!")

best_pt = find_best_pt(run_dir)
if best_pt is None:
    raise RuntimeError("best.pt not found in run dir!")

print("[INFO] Using best.pt:", best_pt)

# Load best.pt model
best_model = YOLO(best_pt)

# Target ONNX path
onnx_path = run_dir / "weights" / ONNX_NAME

print("[INFO] Exporting to ONNX:", onnx_path)
# Export to ONNX (Ultralytics will write default best.onnx into weights)
best_model.export(format="onnx", opset=12, dynamic=True)

# If default ONNX filename differs, rename to desired name
default_onnx = run_dir / "weights" / "best.onnx"
if default_onnx.exists() and default_onnx != onnx_path:
    default_onnx.rename(onnx_path)

print("[OK] ONNX saved at:", onnx_path)


In [ ]:
# Quantize the exported ONNX model to INT8
run_dir = get_latest_run(PROJECT)
weights_dir = run_dir / "weights"
onnx_path = weights_dir / ONNX_NAME
onnx_int8_path = weights_dir / ONNX_INT8_NAME

print("[INFO] Input model :", onnx_path)
print("[INFO] Output model:", onnx_int8_path)

# Perform dynamic quantization
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(onnx_int8_path),
    weight_type=QuantType.QUInt8,
)

print("[OK] Quantized model saved to:", onnx_int8_path)


In [ ]:

# Find the latest run directory
def get_latest_run(project: str) -> Path | None:
    p = Path(project)
    if not p.exists():
        return None
    runs = [d for d in p.iterdir() if d.is_dir()]
    if not runs:
        return None
    return sorted(runs, key=lambda x: x.stat().st_mtime)[-1]

# Find best.pt in run directory
def find_best_pt(run_dir: Path) -> Path | None:
    best = run_dir / "weights" / "best.pt"
    return best if best.exists() else None

# Latest run
run_dir = get_latest_run("runs/train")
if run_dir is None:
    raise RuntimeError("No run directory found!")

best_pt = find_best_pt(run_dir)
if best_pt is None:
    raise RuntimeError("best.pt not found in run directory!")

print("[INFO] Using best.pt:", best_pt)

# Load model
model = YOLO(best_pt)

# TFLite output path
tflite_path = run_dir / "weights" / "best_box_detector.tflite"

# Export to TFLite
print("[INFO] Exporting to TFLite:", tflite_path)
model.export(format="tflite")

print("[OK] TFLite model saved at:", tflite_path)
